# 기획자 관점 — 뉴비 선정 기준 (행동 기반 K-means 군집)

**원본 위치**: `이터널리턴_코드_정리.ipynb` cell 16~18 (섹션: `# 뉴비선정기준(행동군집_K-means)`)

**추정 담당**: 박근우 (확실하지 않음, 팀원 확인 필요)
> 근거: KPT 회고에서 박근우가 "뉴비 유저를 어떻게 정의할 것인가에 대한 기준을 직접 세운 점"을 본인의 핵심 성과로 서술함. 다만 이는 정황적 추정이며 코드에 작성자 표기는 없습니다.

- 관련 문서: `../../docs/03_issues_and_troubleshooting.md` #1, #7, #8, #14
- 입력 파일: `EternalReturn_kakaogames_2024_character_added.csv`


> ### 실행 정보
>
> 이 노트북의 출력은 **이번 정리 과정에서 실제로 실행해 얻은 것**입니다
> (Python 3.11 · pandas 3.0 · scikit-learn 1.9 · shap 0.51 · xgboost 3.2).
> 원본은 Colab 에서 돌았고 분리 시점에 출력이 초기화돼 있었습니다.
>
> 데이터 경로를 `/content/` → `../../data/` 로, 폰트를 나눔고딕 → 맑은 고딕으로 바꿨습니다.
> 입력 데이터는 저장소에 없습니다 — [`data/README.md`](../../data/README.md) 참조.


## 유저 단위 행동 지표 집계 + K-means 군집화

핵심 트러블슈팅 포인트(코드에 그대로 드러남):
- `matchingTeamMode==3`(일반 스쿼드)만 사용 — 코발트 프로토콜 제외 (#1)
- `cluster_base = user_df[user_df["games"]>=2]` — 1게임 유저 제외 (#14)
- 실루엣 점수 k=2~7 비교 후 **k=4를 의도적으로 채택**(통계적 최적은 k=2였지만 기획 실무상 세분화가 더 유용, #7)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

df = pd.read_csv("../../data/EternalReturn_kakaogames_2024_character_added.csv")

# 일반 스쿼드만 사용
squad = df[df["matchingTeamMode"] == 3].copy()

squad["earlyLeave"] = (squad["playTime"] < 600).astype(int)
squad["survive10"] = (squad["playTime"] >= 600).astype(int)
squad["routeNotSelected"] = (
    (squad["routeIdOfStart"] == 0) | (squad["routeIdOfStart"] == -1)
).astype(int)


user_df = squad.groupby("userNum").agg(
    nickname=("nickname", "first"),
    games=("gameId", "count"),
    avg_playTime=("playTime", "mean"),
    earlyLeaveRate=("earlyLeave", "mean"),
    survive10Rate=("survive10", "mean"),
    winRate=("victory", "mean"),
    avg_rank=("gameRank", "mean"),
    avg_kill=("playerKill", "mean"),
    avg_teamKill=("teamKill", "mean"),
    avg_monsterKill=("monsterKill", "mean"),
    avg_craftUncommon=("craftUncommon", "mean"),
    avg_craftRare=("craftRare", "mean"),
    avg_craftEpic=("craftEpic", "mean"),
    avg_hyperLoop=("useHyperLoop", "mean"),
    avg_securityConsole=("useSecurityConsole", "mean"),
    avg_reconDrone=("useReconDrone", "mean"),
    routeNotSelectedRate=("routeNotSelected", "mean"),
    giveUpRate=("giveUp", "mean"),
    accountLevel=("accountLevel", "max"),
    rankPoint=("rankPoint", "max")
).reset_index()

# 1판 유저는 패턴 안정성이 낮아서 우선 제외
cluster_base = user_df[user_df["games"] >= 2].copy()

features = [
    "games",
    "avg_playTime",
    "earlyLeaveRate",
    "survive10Rate",
    "winRate",
    "avg_rank",
    "avg_kill",
    "avg_teamKill",
    "avg_monsterKill",
    "avg_craftUncommon",
    "avg_craftRare",
    "avg_craftEpic",
    "avg_hyperLoop",
    "avg_securityConsole",
    "avg_reconDrone",
    "routeNotSelectedRate",
    "giveUpRate"
]

X = cluster_base[features].replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("=== 군집 수별 실루엣 점수 ===")
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    print(k, round(silhouette_score(X_scaled, labels), 4))

# 4개 군집으로 실행
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
cluster_base["cluster"] = kmeans.fit_predict(X_scaled)

summary = cluster_base.groupby("cluster").agg(
    users=("userNum", "count"),
    avg_games=("games", "mean"),
    avg_accountLevel=("accountLevel", "mean"),
    avg_rankPoint=("rankPoint", "mean"),
    avg_playTime=("avg_playTime", "mean"),
    earlyLeaveRate=("earlyLeaveRate", "mean"),
    survive10Rate=("survive10Rate", "mean"),
    winRate=("winRate", "mean"),
    avg_rank=("avg_rank", "mean"),
    avg_kill=("avg_kill", "mean"),
    avg_monsterKill=("avg_monsterKill", "mean"),
    avg_craftRare=("avg_craftRare", "mean"),
    avg_hyperLoop=("avg_hyperLoop", "mean"),
    avg_securityConsole=("avg_securityConsole", "mean"),
    routeNotSelectedRate=("routeNotSelectedRate", "mean"),
    giveUpRate=("giveUpRate", "mean")
).reset_index()

print("\n=== 군집별 요약 ===")
display(summary.sort_values("earlyLeaveRate", ascending=False))

cluster_base.to_csv("../../data/user_behavior_clusters.csv", index=False, encoding="utf-8-sig")
summary.to_csv("../../data/user_cluster_summary.csv", index=False, encoding="utf-8-sig")

=== 군집 수별 실루엣 점수 ===
2 0.1915
3 0.1398
4 0.1448
5 0.1411
6 0.1181
7 0.1086

=== 군집별 요약 ===
   cluster  users  ...  routeNotSelectedRate  giveUpRate
0        0  13369  ...              0.181373    0.000264
3        3    401  ...              0.173130    0.412350
1        1  21193  ...              0.215342    0.000592
2        2  10912  ...              0.273675    0.000319

[4 rows x 17 columns]


## ⚠ 아래 셀은 원본부터 미완성/단편 상태입니다

`profile_long`이라는 DataFrame을 참조하지만, 이를 생성하는 코드가 원본 노트북에 없습니다. 변수명(`feature`, `z_score`, `abs_z`, `cluster`)과 부록 슬라이드 21의 "Cluster 0~3 주요 행동패턴(z=...)" 표를 대조해 보면, 아마도 위 셀에서 만든 `cluster_base`/`summary`를 이용해 **군집별 평균을 표준화(z-score)한 long-format 테이블**을 만드는 코드가 이 사이에 있었을 것으로 추정됩니다. 예를 들면 아래와 같은 형태였을 가능성이 있습니다 (검증되지 않은 추정 재구성, 실행하지 마세요):

```python
# 추정 재구성 (미검증) — 실제 원본 코드 아님
# feat_cols = features  # 위 셀의 17개 피처
# means = cluster_base.groupby('cluster')[feat_cols].mean()
# z = (means - means.mean()) / means.std()
# profile_long = z.reset_index().melt(id_vars='cluster', var_name='feature', value_name='z_score')
# profile_long['abs_z'] = profile_long['z_score'].abs()
```
자세한 내용은 `../../docs/03_issues_and_troubleshooting.md`에 별도 정리하지 않았다면 이슈로 등록해 팀원에게 확인하세요.

> ### 🐛 `profile_long` 이 원본 어디에도 정의돼 있지 않습니다
>
> 아래 셀은 `profile_long` 을 **쓰기만** 하는데, 원본 통합 노트북
> (`원본/이터널리턴 코드 정리.ipynb`)을 전수 검색해도 **정의하는 코드가 없습니다.**
> 그대로 실행하면 `NameError: name 'profile_long' is not defined` 가 납니다.
>
> 사용 방식(`cluster` · `feature` · `z_score` · `abs_z` 4개 컬럼)과
> [`docs/03_issues_and_troubleshooting.md`](../../docs/03_issues_and_troubleshooting.md) #8 의
> *"`cluster_gap`(피처별 군집간 z-score 격차)"* 서술로 미루어,
> **군집별 요약(`summary`)을 피처 축으로 z-표준화한 뒤 롱포맷으로 편 것**으로 판단됩니다.
>
> 아래 복원 셀이 그 정의입니다. **원본 코드가 아니라 이번에 재구성한 것**이며,
> 팀이 의도한 계산과 다를 수 있습니다 — 확인 후 갱신해 주세요.


In [ ]:
# 🔧 복원(포트폴리오 정리) — 원본에 정의가 없어 사용처 스키마로부터 재구성
#    필요 컬럼: cluster · feature · z_score · abs_z
feat_cols = [c for c in summary.columns if c not in ('cluster', 'users')]
_z = summary.set_index('cluster')[feat_cols].astype(float)
_z = (_z - _z.mean()) / _z.std(ddof=0)          # 피처별로 군집 간 z-표준화

profile_long = (_z.reset_index()
                  .melt(id_vars='cluster', var_name='feature', value_name='z_score'))
profile_long['abs_z'] = profile_long['z_score'].abs()

print(f'profile_long 복원: {profile_long.shape}  '
      f'(군집 {profile_long["cluster"].nunique()} × 피처 {profile_long["feature"].nunique()})')
display(profile_long.head())


profile_long 복원: (60, 4)  (군집 4 × 피처 15)
   cluster           feature   z_score     abs_z
0        0         avg_games -0.525586  0.525586
1        1         avg_games  1.712755  1.712755
2        2         avg_games -0.386771  0.386771
3        3         avg_games -0.800398  0.800398
4        0  avg_accountLevel -0.250685  0.250685


In [ ]:
for c in sorted(profile_long["cluster"].unique()):
    print(f"\n=== Cluster {c} 주요 행동패턴 ===")
    tmp = profile_long[profile_long["cluster"] == c].copy()
    tmp = tmp.sort_values("abs_z", ascending=False).head(8)

    for _, r in tmp.iterrows():
        direction = "높음" if r["z_score"] > 0 else "낮음"
        print(f"{r['feature']:25s} {direction:4s} z={r['z_score']:.2f}")


=== Cluster 0 주요 행동패턴 ===
earlyLeaveRate            높음   z=1.54
survive10Rate             낮음   z=-1.54
avg_playTime              낮음   z=-1.35
avg_rank                  높음   z=1.26
avg_monsterKill           낮음   z=-1.12
avg_securityConsole       낮음   z=-1.02
avg_kill                  낮음   z=-0.90
avg_hyperLoop             낮음   z=-0.88

=== Cluster 1 주요 행동패턴 ===
avg_games                 높음   z=1.71
survive10Rate             높음   z=0.69
earlyLeaveRate            낮음   z=-0.69
avg_craftRare             높음   z=0.63
giveUpRate                낮음   z=-0.58
avg_rankPoint             높음   z=0.56
winRate                   낮음   z=-0.50
avg_hyperLoop             높음   z=0.38

=== Cluster 2 주요 행동패턴 ===
winRate                   높음   z=1.71
avg_kill                  높음   z=1.68
routeNotSelectedRate      높음   z=1.59
avg_securityConsole       높음   z=1.58
avg_monsterKill           높음   z=1.55
avg_rank                  낮음   z=-1.51
avg_hyperLoop             높음   z=1.47
avg_playTime              높음   z=1.